[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sausterm/omniscience-research/blob/main/notebooks/dti_demo.ipynb)

# Curvature Anisotropy for Diffusion Tensor Imaging
## A Novel Metric from the Riemannian Geometry of Sym$^+$(3)

**OmniSciences LLC** | [omnisciences.io](https://omnisciences.io)

---

This notebook demonstrates a new DTI analysis metric — **Curvature Anisotropy (CA)** — that detects white matter structure where standard Fractional Anisotropy (FA) fails.

All computation runs on the OmniSciences API. No local installation required.

> **API Access**: Email sloan@omnisciences.io for a free academic API key.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

# ── Configuration ──────────────────────────────────────────────
API_URL = "https://dti.omnisciences.io/dti"
API_KEY = "YOUR_API_KEY"  # Email sloan@omnisciences.io for API access
# ──────────────────────────────────────────────────────────────

HEADERS = {"X-API-Key": API_KEY, "Content-Type": "application/json"}


def dti_analyze(tensor):
    """Send a single 3x3 tensor to the API, get back all metrics."""
    resp = requests.post(f"{API_URL}/analyze", json={"tensor": np.asarray(tensor).tolist()}, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


def dti_distance(tensor1, tensor2):
    """Geodesic distance between two tensors."""
    resp = requests.post(f"{API_URL}/distance", json={
        "tensor1": np.asarray(tensor1).tolist(),
        "tensor2": np.asarray(tensor2).tolist()
    }, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


def dti_interpolate(tensor1, tensor2, t):
    """Geodesic interpolation at parameter t in [0,1]."""
    resp = requests.post(f"{API_URL}/interpolate", json={
        "tensor1": np.asarray(tensor1).tolist(),
        "tensor2": np.asarray(tensor2).tolist(),
        "t": t
    }, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


def dti_batch(tensors):
    """Batch analyze an array of tensors."""
    resp = requests.post(f"{API_URL}/batch", json={
        "tensors": [np.asarray(t).tolist() for t in tensors]
    }, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


# Quick connectivity check
try:
    r = requests.get(f"{API_URL}/health", headers=HEADERS, timeout=5)
    print(f"API status: {r.json().get('status', 'unknown')}")
    print(f"Geometry: {r.json().get('geometry', {})}")
except Exception as e:
    print(f"Cannot reach API at {API_URL}")
    print(f"Error: {e}")
    print("\nTo get an API key, email sloan@omnisciences.io")

## 1. Standard DTI Metrics

Let's verify the API reproduces expected clinical values for canonical tissue types.

In [ ]:
# Canonical diffusion tensors (eigenvalues in mm²/s)
WM  = [[1.7e-3, 0, 0], [0, 0.3e-3, 0], [0, 0, 0.3e-3]]     # White matter
GM  = [[1.0e-3, 0, 0], [0, 0.7e-3, 0], [0, 0, 0.7e-3]]     # Gray matter
CSF = [[3.0e-3, 0, 0], [0, 3.0e-3, 0], [0, 0, 3.0e-3]]     # CSF

tissues = {"White Matter": WM, "Gray Matter": GM, "CSF": CSF}

print(f"{'Tissue':<15} {'FA':>8} {'CA':>8} {'MD (mm²/s)':>12} {'Tissue Class':>14}")
print("=" * 60)
for name, D in tissues.items():
    r = dti_analyze(D)
    print(f"{name:<15} {r['fa']:>8.3f} {r['curvature_anisotropy']:>8.3f} "
          f"{r['md']:>12.2e} {r['tissue_type']:>14}")

## 2. The Killer Feature: Fiber Crossings

In ~90% of white matter voxels, multiple fiber bundles cross. FA collapses at these crossings — a dangerous false negative. **Curvature Anisotropy stays elevated.**

In [ ]:
# Two perpendicular fiber bundles
fiber_x = [[1.7e-3, 0, 0], [0, 0.3e-3, 0], [0, 0, 0.3e-3]]
fiber_y = [[0.3e-3, 0, 0], [0, 1.7e-3, 0], [0, 0, 0.3e-3]]

# 90° crossing = average of the two (what a voxel measures)
crossing = (np.array(fiber_x) + np.array(fiber_y)) / 2

# Damaged white matter (real pathology)
damaged = [[1.0e-3, 0, 0], [0, 0.6e-3, 0], [0, 0, 0.6e-3]]

configs = {
    "Single fiber (x)":   fiber_x,
    "Single fiber (y)":   fiber_y,
    "90° fiber crossing": crossing.tolist(),
    "Damaged WM":         damaged,
    "CSF (isotropic)":    CSF,
}

results = {}
print(f"{'Configuration':<22} {'FA':>7} {'CA':>7} {'MD':>10}  Interpretation")
print("=" * 78)
for name, D in configs.items():
    r = dti_analyze(D)
    results[name] = r
    
    if name == "90° fiber crossing":
        interp = "⚠ FA FAILS — CA detects intact structure"
    elif name == "Damaged WM":
        interp = "Both FA and CA reduced — real damage"
    elif "fiber" in name.lower():
        interp = "Healthy fiber — both metrics high"
    else:
        interp = "Isotropic — both metrics zero"
    
    print(f"{name:<22} {r['fa']:>7.3f} {r['curvature_anisotropy']:>7.3f} {r['md']:>10.2e}  {interp}")

In [ ]:
# Visualize FA vs CA
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = list(results.keys())
fa_vals = [results[n]['fa'] for n in names]
ca_vals = [results[n]['curvature_anisotropy'] for n in names]
colors = ['#2ecc71', '#2ecc71', '#e74c3c', '#f39c12', '#3498db']

axes[0].barh(names, fa_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Fractional Anisotropy (FA)')
axes[0].set_title('Standard FA — Misses Crossings')
axes[0].axvline(x=0.3, color='gray', linestyle='--', alpha=0.5, label='WM threshold')
axes[0].set_xlim(0, 1)
axes[0].legend()

axes[1].barh(names, ca_vals, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Curvature Anisotropy (CA)')
axes[1].set_title('Curvature Anisotropy — Detects Crossings')
axes[1].set_xlim(0, max(ca_vals) * 1.2 if ca_vals else 1)

plt.tight_layout()
plt.suptitle('FA vs Curvature Anisotropy at Fiber Crossings', y=1.02, fontsize=14, fontweight='bold')
plt.show()

print("\n→ At the 90° crossing (red bar), FA drops to near zero.")
print("  CA remains elevated — correctly detecting intact fiber structure.")

## 3. Pathology Discrimination

Can your DTI tool distinguish **demyelination** (treatable) from **edema** (different treatment)?

FA cannot. CA can.

In [ ]:
# Simulate pathological tensors
normal_wm     = [[1.7e-3, 0, 0], [0, 0.3e-3, 0], [0, 0, 0.3e-3]]
demyelination = [[1.7e-3, 0, 0], [0, 0.7e-3, 0], [0, 0, 0.7e-3]]  # Radial diffusivity ↑
edema         = [[2.1e-3, 0, 0], [0, 0.9e-3, 0], [0, 0, 0.9e-3]]  # Isotropic swelling
axonal_loss   = [[0.8e-3, 0, 0], [0, 0.5e-3, 0], [0, 0, 0.5e-3]]  # Axial diffusivity ↓

pathologies = {
    "Normal WM":     normal_wm,
    "Demyelination": demyelination,
    "Edema":         edema,
    "Axonal Loss":   axonal_loss,
}

path_results = {}
print(f"{'Pathology':<18} {'FA':>7} {'CA':>7} {'MD':>10}")
print("=" * 45)
for name, D in pathologies.items():
    r = dti_analyze(D)
    path_results[name] = r
    print(f"{name:<18} {r['fa']:>7.3f} {r['curvature_anisotropy']:>7.3f} {r['md']:>10.2e}")

print(f"\nFA  |demyel - edema| = {abs(path_results['Demyelination']['fa'] - path_results['Edema']['fa']):.3f}")
print(f"CA  |demyel - edema| = {abs(path_results['Demyelination']['curvature_anisotropy'] - path_results['Edema']['curvature_anisotropy']):.3f}")

In [ ]:
# FA vs CA scatter for pathologies
fig, ax = plt.subplots(figsize=(8, 6))

color_map = {'Normal WM': '#2ecc71', 'Demyelination': '#e67e22',
             'Edema': '#e74c3c', 'Axonal Loss': '#9b59b6'}

for name, r in path_results.items():
    ax.scatter(r['fa'], r['curvature_anisotropy'], s=200, c=color_map[name],
              edgecolors='black', linewidth=1.5, label=name, zorder=5)

ax.axvspan(0.3, 0.55, alpha=0.1, color='red', label='FA ambiguity zone')
ax.set_xlabel('Fractional Anisotropy (FA)', fontsize=13)
ax.set_ylabel('Curvature Anisotropy (CA)', fontsize=13)
ax.set_title('Pathology Discrimination: FA vs CA', fontsize=14, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print("→ Demyelination and Edema overlap on the FA axis (x) but separate on CA (y).")

## 4. Geodesic Interpolation

Standard tools use Euclidean interpolation, which causes **tensor swelling** (non-physical artifact). Our geodesic interpolation follows the true shortest path on the curved manifold — always producing valid tensors.

In [ ]:
# Interpolate WM → GM along the geodesic
n_steps = 20
t_vals = np.linspace(0, 1, n_steps)

geo_fa, geo_ca, geo_det = [], [], []
euc_fa, euc_det = [], []

WM_arr, GM_arr = np.array(WM), np.array(GM)

for t in t_vals:
    # Geodesic (via API)
    r = dti_interpolate(WM, GM, float(t))
    D_geo = np.array(r['tensor'])
    r_metrics = dti_analyze(D_geo.tolist())
    geo_fa.append(r_metrics['fa'])
    geo_ca.append(r_metrics['curvature_anisotropy'])
    geo_det.append(np.linalg.det(D_geo))
    
    # Euclidean (local, no API needed — it's just linear averaging)
    D_euc = (1 - t) * WM_arr + t * GM_arr
    euc_eigs = np.linalg.eigvalsh(D_euc)
    md_e = np.mean(euc_eigs)
    euc_fa.append(float(np.sqrt(1.5) * np.linalg.norm(euc_eigs - md_e) / np.linalg.norm(euc_eigs)))
    euc_det.append(np.linalg.det(D_euc))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(t_vals, geo_fa, 'b-', lw=2, label='Geodesic (ours)')
axes[0].plot(t_vals, euc_fa, 'r--', lw=2, label='Euclidean (FSL)')
axes[0].set_ylabel('FA'); axes[0].set_title('Fractional Anisotropy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_xlabel('t (WM → GM)')

axes[1].plot(t_vals, geo_ca, 'b-', lw=2.5)
axes[1].set_ylabel('CA'); axes[1].set_title('Curvature Anisotropy (ours only)')
axes[1].set_xlabel('t (WM → GM)'); axes[1].grid(True, alpha=0.3)

max_det = max(np.linalg.det(WM_arr), np.linalg.det(GM_arr))
axes[2].plot(t_vals, geo_det, 'b-', lw=2, label='Geodesic')
axes[2].plot(t_vals, euc_det, 'r--', lw=2, label='Euclidean')
axes[2].axhline(y=max_det, color='gray', ls=':', alpha=0.7, label='max endpoint det')
axes[2].set_ylabel('det(D)'); axes[2].set_title('Swelling Check')
axes[2].set_xlabel('t (WM → GM)'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Geodesic vs Euclidean Interpolation', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Geodesic Distance Matrix

The Riemannian distance between tensors respects the curved geometry of Sym$^+$(3) — unlike Euclidean or log-Euclidean approximations used by other tools.

In [ ]:
tissue_list = [("White Matter", WM), ("Gray Matter", GM), ("CSF", CSF)]

print("GEODESIC DISTANCE MATRIX")
print(f"{'':>15}", end="")
for name, _ in tissue_list:
    print(f" {name:>15}", end="")
print()

for i, (name_i, Di) in enumerate(tissue_list):
    print(f"{name_i:>15}", end="")
    for j, (name_j, Dj) in enumerate(tissue_list):
        if i == j:
            print(f" {'0.0000':>15}", end="")
        else:
            r = dti_distance(Di, Dj)
            print(f" {r['distance']:>15.4f}", end="")
    print()

print("\n→ CSF is furthest from everything (high MD, isotropic)")
print("→ Distances satisfy the triangle inequality (guaranteed by Riemannian geometry)")

## 6. Try Your Own Tensors

Paste in a 3×3 symmetric positive-definite matrix from your DTI data:

In [ ]:
# ── Replace this with your own tensor ──────────────────────────
my_tensor = [
    [1.2e-3, 0.3e-3, 0.0],
    [0.3e-3, 0.8e-3, 0.1e-3],
    [0.0,    0.1e-3, 0.5e-3]
]
# ──────────────────────────────────────────────────────────────

r = dti_analyze(my_tensor)

print("ANALYSIS RESULTS")
print("=" * 45)
print(f"  Fractional Anisotropy (FA):  {r['fa']:.4f}")
print(f"  Curvature Anisotropy (CA):   {r['curvature_anisotropy']:.4f}")
print(f"  Mean Diffusivity (MD):       {r['md']:.2e} mm²/s")
print(f"  Tissue Classification:       {r['tissue_type']}")
print(f"  Confidence:                  {r['confidence']:.1%}")
print(f"  Eigenvalues:                 {r['eigenvalues']}")

---

## Summary

| Feature | omni-toolkit | FSL | DIPY | FreeSurfer |
|---------|:---:|:---:|:---:|:---:|
| **Curvature Anisotropy** | **Yes** | No | No | No |
| **V+/V- decomposition** | **Yes** | No | No | No |
| Geodesic distance | Yes | No | Partial | No |
| Swelling-free interpolation | Yes | No | Partial | No |
| Standard FA/MD | Yes | Yes | Yes | Yes |

### API Access

Email **sloan@omnisciences.io** for a free academic API key.

- **Free tier**: 100 requests/month (academic)
- **Professional tier**: 50,000 requests/month
- **Enterprise**: On-premise deployment, custom models

### Contact

**OmniSciences LLC**
Email: sloan@omnisciences.io
Web: [omnisciences.io](https://omnisciences.io)

*Patent pending. © 2026 OmniSciences LLC.*